# b) State-level emissions and real GDP

Estimate the association between natural log sector-99 emissions (`sector_99_emissions_tons`) and natural log real GDP (`real_gdp_millions_2017usd`) separately for each pollutant across state-years:

1. Pooled OLS with an intercept and no fixed effects.
2. OLS with state fixed effects.
3. OLS with state and year fixed effects.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

DATA_PATH = Path("NEI_clean_state.parquet")
TABLE_PATH = Path("emissions_gdp_regressions.tex")
EMISSIONS = "sector_99_emissions_tons"
GDP = "real_gdp_millions_2017usd"
df = pd.read_parquet(DATA_PATH)
required = ["state", "year", "pollutant", EMISSIONS, GDP]
assert set(required).issubset(df.columns), "Missing required columns."
assert df[["state", "year", "pollutant"]].notna().all().all()
assert not df.duplicated(["state", "year", "pollutant"]).any()
assert df.groupby(["state", "year"])[GDP].nunique(dropna=False).le(1).all()

# Exclude missing, nonfinite, or nonpositive outcomes/GDP before taking logs.
# Use the same eligible sample for all three specifications within each pollutant.
# Verify >0 for logs:
valid = (np.isfinite(df[EMISSIONS]) & df[EMISSIONS].gt(0)
         & np.isfinite(df[GDP]) & df[GDP].gt(0))
coverage = df.assign(eligible=valid).groupby("pollutant").agg(
    input_rows=("year", "size"), eligible_rows=("eligible", "sum")
)
coverage["excluded_rows"] = coverage["input_rows"] - coverage["eligible_rows"]
display(coverage)
assert coverage["eligible_rows"].gt(0).all()
sample = df.loc[valid, required].copy()
# Plain Python-compatible dtypes work with Patsy's categorical encoding.
sample["state"] = sample["state"].astype(str)
sample["year"] = sample["year"].astype(int)
sample["pollutant"] = sample["pollutant"].astype(str)
sample["log_emissions"] = np.log(sample[EMISSIONS])
sample["log_gdp"] = np.log(sample[GDP])
print("Reporting years:", sorted(sample["year"].unique().tolist()))
display(sample.groupby("pollutant").agg(
    observations=("year", "size"), states=("state", "nunique"), years=("year", "nunique")
))

## Estimate the three specifications

State and year effects enter as categorical indicators. The intercept and individual fixed-effect coefficients are omitted from the table. Each panel reports the GDP slope, its clustered standard error, observations, state clusters, and overall R².

In [ ]:
specifications = {
    "(1) No FE": "log_emissions ~ log_gdp",
    "(2) State FE": "log_emissions ~ log_gdp + C(state)",
    "(3) State + year FE": "log_emissions ~ log_gdp + C(state) + C(year)",
}
models = {}
records = []
for pollutant, group in sample.groupby("pollutant", sort=True):
    assert group["state"].nunique() > 1
    for specification, formula in specifications.items():
        model = smf.ols(formula, data=group, missing="raise").fit(
            cov_type="cluster",
            cov_kwds={"groups": group["state"], "use_correction": True,
                      "df_correction": True},
            use_t=True,
        )
        # Check that rank is equal to regressors and number of obs is right
        assert np.linalg.matrix_rank(model.model.exog) == model.model.exog.shape[1]
        assert int(model.nobs) == len(group)
        models[(pollutant, specification)] = model
        records.append({
            "pollutant": pollutant, "specification": specification,
            "coefficient": model.params["log_gdp"],
            "std_error": model.bse["log_gdp"],
            "p_value": model.pvalues["log_gdp"],
            "observations": int(model.nobs),
            "states": group["state"].nunique(), "r_squared": model.rsquared,
        })
results = pd.DataFrame(records).set_index(["pollutant", "specification"])
assert results.groupby(level="pollutant")["observations"].nunique().eq(1).all()

In [ ]:
# Build one table with a panel for each pollutant and a column per specification.
def stars(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""

rows = []
row_index = []
for pollutant in sorted(sample["pollutant"].unique()):
    panel = results.loc[pollutant].reindex(specifications)
    panel_rows = {
        "Log real GDP": [f"{r.coefficient:.3f}{stars(r.p_value)}" for r in panel.itertuples()],
        "Clustered SE": [f"({r.std_error:.3f})" for r in panel.itertuples()],
        "Observations": [f"{int(n):,}" for n in panel["observations"]],
        "State clusters": [str(int(n)) for n in panel["states"]],
        "R² (overall)": [f"{r:.3f}" for r in panel["r_squared"]],
        "State FE": ["No", "Yes", "Yes"],
        "Year FE": ["No", "No", "Yes"],
    }
    for label, values in panel_rows.items():
        row_index.append((pollutant, label))
        rows.append(values)
regression_table = pd.DataFrame(
    rows, index=pd.MultiIndex.from_tuples(row_index, names=["Pollutant", "Statistic"]),
    columns=list(specifications),
)
display(regression_table)
print("State-clustered standard errors in parentheses; * p<0.10, ** p<0.05, *** p<0.01.")

## Export for LaTeX

The file is a complete `table` environment with pollutant panels and notes. In the document preamble add `\usepackage{booktabs}`, then insert the table with:

```latex
\input{emissions_gdp_regressions.tex}
```

The output path is relative to this notebook's working directory. Rerunning this cell replaces the generated table.

In [ ]:
pollutant_labels = {"NOX": r"NO$_x$", "PM25": r"PM$_{2.5}$", "SO2": r"SO$_2$", "VOC": "VOCs"}

def tex_escape(value):
    replacements = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
                    "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",
                    "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}
    return "".join(replacements.get(char, char) for char in str(value))

lines = [
    r"\begin{table}[htbp]", r"\centering",
    r"\caption{State-level emissions and real GDP}",
    r"\label{tab:emissions-gdp}", r"\small",
    r"\begin{tabular}{lccc}", r"\toprule",
    r" & (1) & (2) & (3) \\",
    r" & No FE & State FE & State + year FE \\",
    r"\midrule",
]
for i, pollutant in enumerate(sorted(sample["pollutant"].unique())):
    if i:
        lines.append(r"\midrule")
    label = pollutant_labels.get(pollutant, tex_escape(pollutant))
    lines.append(r"\multicolumn{4}{l}{\textbf{Panel " + chr(65 + i) + ": " + label + r"}} \\")
    for statistic, row in regression_table.loc[pollutant].iterrows():
        label = r"$R^2$ (overall)" if statistic == "R² (overall)" else tex_escape(statistic)
        lines.append(label + " & " + " & ".join(row) + r" \\")
lines.extend([
    r"\bottomrule", r"\end{tabular}", r"\par\smallskip",
    r"\begin{minipage}{0.95\linewidth}\footnotesize",
    r"\textit{Notes:} Dependent variable: natural log sector-99 emissions in tons. "
    r"The regressor is natural log real GDP in millions of 2017 US dollars. "
    r"Unweighted OLS, estimated separately by pollutant. "
    + f"Reporting years: {', '.join(map(str, sorted(sample['year'].unique())))}. "
    + r"DC is treated as a state. Standard errors in parentheses are clustered by state "
    r"with a finite-sample correction; p-values use a t distribution with the number "
    r"of state clusters minus one degrees of freedom. "
    r"$R^2$ includes the fixed effects. All specifications within a pollutant use the same sample. "
    r"Missing, nonfinite, or nonpositive emissions/GDP are excluded before taking logs. "
    + f"Total excluded pollutant-state-year rows: {int(coverage['excluded_rows'].sum())}. "
    + r"* $p<0.10$, ** $p<0.05$, *** $p<0.01$.",
    r"\end{minipage}", r"\end{table}",
])
TABLE_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Saved table to {TABLE_PATH.resolve()}")